In [ ]:
# Treino CNN - ResNet50, EfficientNet-B0, ConvNeXt-Tiny e DenseNet121 (P2 · Tema F)
# Fine-tuning de quatro CNN pré-treinadas (ImageNet) no FieldVitis, com o `splits.json` do repositório (3 seeds).

In [1]:
# --- Configuração: dados (Drive) + splits (GitHub) ---
from google.colab import drive
drive.mount("/content/drive")

!git clone -q https://github.com/franciscorafael-dotcom/DL_P2_TP2_G2-F1.git
!mkdir -p /content/fieldvitis && unzip -q -o /content/drive/MyDrive/fieldvitis.zip -d /content/fieldvitis/

import json
from pathlib import Path
DATA_DIR = Path("/content/fieldvitis")
REPO_DIR = Path("/content/DL_P2_TP2_G2-F1")

# Verificação: as imagens batem com o splits.json?
sj = json.load(open(REPO_DIR / "data/splits.json"))
CLASSES = sj["classes"]
print("Pastas:      ", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))
print("splits.json: ", CLASSES)
for s in ["0", "1", "2"]:
    sp = sj["seeds"][s]
    faltam = [p for k in sp for p in sp[k] if not (DATA_DIR / p).exists()]
    print(f"seed {s}:", {k: len(v) for k, v in sp.items()}, "| em falta:", len(faltam))

Mounted at /content/drive
Pastas:       ['Black_rot', 'Esca', 'Healthy', 'Leaf_blight']
splits.json:  ['Black_rot', 'Esca', 'Healthy', 'Leaf_blight']
seed 0: {'train': 388, 'val': 84, 'test': 84} | em falta: 0
seed 1: {'train': 388, 'val': 84, 'test': 84} | em falta: 0
seed 2: {'train': 388, 'val': 84, 'test': 84} | em falta: 0


In [2]:
# Configuração e Carregamento das imagens
import time, gc
import numpy as np, pandas as pd
import tensorflow as tf, keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import f1_score, recall_score

IMG_SIZE, BATCH = 224, 32
EPOCHS_HEAD, LR_HEAD = 5, 1e-3     # fase 1: só a cabeça (base congelada)
EPOCHS_FT, LR_FT = 25, 1e-5        # fase 2: fine-tuning da rede toda
PATIENCE = 6                       # early stopping no Macro-F1 de validação
SEEDS = [0, 1, 2]
RUNS_DIR = Path("/content/drive/MyDrive/P2_runs"); RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Keras", keras.__version__, "| backend", keras.backend.backend(),
      "| GPU:", tf.config.list_physical_devices("GPU"))

# Todas as imagens carregadas uma vez: RGB, orientação EXIF corrigida, 224x224
all_paths = sorted({p for s in sj["seeds"].values() for k in ("train", "val", "test") for p in s[k]})
cls_idx = {c: i for i, c in enumerate(CLASSES)}
IMGS, LABELS = {}, {}
for p in all_paths:
    im = ImageOps.exif_transpose(Image.open(DATA_DIR / p)).convert("RGB")
    IMGS[p] = np.asarray(im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
    LABELS[p] = cls_idx[Path(p).parts[0]]
print("Imagens carregadas:", len(IMGS), "| por classe:", np.bincount(list(LABELS.values())))

Keras 3.13.2 | backend tensorflow | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Imagens carregadas: 556 | por classe: [ 39 201 158 158]


In [3]:
# Cada rede espera os píxeis num formato diferente (o mesmo usado no treino do ImageNet)
PREPROCESS = {
    "resnet50": keras.applications.resnet50.preprocess_input,           # converte para BGR e subtrai a média do ImageNet
    "efficientnetb0": keras.applications.efficientnet.preprocess_input,  # não altera: o modelo normaliza por dentro
    "convnext_tiny": keras.applications.convnext.preprocess_input,       # não altera: o modelo normaliza por dentro
    "densenet121": keras.applications.densenet.preprocess_input,         # escala para [0,1] e normaliza com média/desvio do ImageNet
}

def preparar_dados(seed, nome):
    """Devolve treino/val/teste da seed indicada, já pré-processados para o modelo."""
    sp = sj["seeds"][str(seed)]
    dados = {}
    for split in ["train", "val", "test"]:
        x = np.stack([IMGS[p] for p in sp[split]]).astype("float32")
        y = np.array([LABELS[p] for p in sp[split]])
        dados[split] = (PREPROCESS[nome](x), y, sp[split])
    return dados

In [4]:
def criar_modelo(nome):
    """Base pré-treinada no ImageNet + augmentation + camada final de 4 classes."""
    if nome == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                           input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                 input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "convnext_tiny":
        base = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet",
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    else:
        base = keras.applications.DenseNet121(include_top=False, weights="imagenet",
                                              input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    base.trainable = False   # começa congelada

    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.2),
    ])  # só atua durante o treino

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = base(x, training=False)   # BatchNorm fica em modo inferência, mesmo no fine-tuning
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs), base

In [5]:
# Treinar e Guardar
def guardar_previsoes(nome, seed, split, caminhos, y, probs):
    """CSV para o avaliador: uma linha por imagem, com a classe verdadeira, a prevista e as 4 probabilidades."""
    df = pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASSES])
    df.insert(0, "y_pred", [CLASSES[i] for i in probs.argmax(axis=1)])
    df.insert(0, "y_true", [CLASSES[i] for i in y])
    df.insert(0, "path", caminhos)
    df.insert(0, "split", split)
    df.insert(0, "seed", seed)
    df.insert(0, "modelo", nome)
    df.to_csv(RUNS_DIR / f"preds_{nome}_seed{seed}_{split}.csv", index=False)

class EarlyStoppingMacroF1(keras.callbacks.Callback):
    """Early stopping no Macro-F1 de validação (maximizar); restaura os melhores pesos no fim."""
    def __init__(self, x_val, y_val, patience):
        super().__init__()
        self.x_val, self.y_val, self.patience = x_val, y_val, patience

    def on_train_begin(self, logs=None):
        self.best, self.wait, self.best_weights = -1.0, 0, None

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.x_val, batch_size=BATCH, verbose=0).argmax(axis=1)
        f1 = f1_score(self.y_val, pred, average="macro")
        print(f"    val_macro_f1 = {f1:.4f} (melhor: {max(self.best, f1):.4f})")
        if f1 > self.best:
            self.best, self.wait = f1, 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

def treinar(nome, seed):
    keras.utils.set_random_seed(seed)   # reprodutibilidade
    dados = preparar_dados(seed, nome)
    x_tr, y_tr, _ = dados["train"]
    x_va, y_va, _ = dados["val"]

    # Pesos por classe: o black rot (poucas imagens) pesa mais na loss
    contagens = np.bincount(y_tr, minlength=len(CLASSES))
    class_weight = {i: len(y_tr) / (len(CLASSES) * n) for i, n in enumerate(contagens)}

    model, base = criar_modelo(nome)

    # Fase 1: treina só a camada final (base congelada)
    model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_HEAD,
              batch_size=BATCH, class_weight=class_weight, verbose=2)

    # Fase 2: fine-tuning de toda a rede, com learning rate baixo
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(LR_FT),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    early = EarlyStoppingMacroF1(x_va, y_va, PATIENCE)
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_FT,
              batch_size=BATCH, class_weight=class_weight, callbacks=[early], verbose=2)

    # Guardar o modelo (só no Drive) e as previsões de val e teste (para o avaliador)
    model.save(RUNS_DIR / f"{nome}_seed{seed}.keras")
    for split in ["val", "test"]:
        x, y, caminhos = dados[split]
        guardar_previsoes(nome, seed, split, caminhos, y, model.predict(x, verbose=0))

    _, val_acc = model.evaluate(x_va, y_va, verbose=0)
    print(f">>> {nome} seed {seed}: accuracy de validação = {val_acc:.3f}")

In [6]:
# ResNet50
for seed in SEEDS:
    print(f"\n===== ResNet50 | seed {seed} =====")
    treinar("resnet50", seed)
    keras.backend.clear_session()   # liberta a memória da GPU entre corridas


===== ResNet50 | seed 0 =====
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/5
13/13 - 18s - 1s/step - accuracy: 0.4175 - loss: 1.3810 - val_accuracy: 0.7619 - val_loss: 0.6491
Epoch 2/5
13/13 - 2s - 160ms/step - accuracy: 0.7113 - loss: 0.8103 - val_accuracy: 0.8452 - val_loss: 0.4531
Epoch 3/5
13/13 - 2s - 161ms/step - accuracy: 0.8067 - loss: 0.5052 - val_accuracy: 0.9048 - val_loss: 0.2826
Epoch 4/5
13/13 - 2s - 159ms/step - accuracy: 0.8737 - loss: 0.3983 - val_accuracy: 0.9286 - val_loss: 0.2476
Epoch 5/5
13/13 - 2s - 159ms/step - accuracy: 0.8866 - loss: 0.3410 - val_accuracy: 0.9524 - val_loss: 0.2122
Epoch 1/25
    val_macro_f1 = 0.9338 (melhor: 0.9338)
13/13 - 33s - 3s/step - accuracy: 0.9227 - loss: 0.2762 - val_accuracy: 0.9643 - val_loss: 0.1036
Epoch 2/25
    val_macro_f1 = 0.9408 (melhor: 0.9408)
13/13 - 5s - 410ms/step - accuracy: 0.9253 - loss: 0.2364 - val_accuracy: 0.9762 - val_loss: 0.0577
Epoch 3/25
    val_macro_f1 = 0.9224 (melhor: 0.9408)
13/13 - 5s

In [7]:
# EfficientNet-B0
for seed in SEEDS:
    print(f"\n===== EfficientNet-B0 | seed {seed} =====")
    treinar("efficientnetb0", seed)
    keras.backend.clear_session()


===== EfficientNet-B0 | seed 0 =====
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/5
13/13 - 15s - 1s/step - accuracy: 0.4536 - loss: 1.1569 - val_accuracy: 0.7381 - val_loss: 0.9386
Epoch 2/5
13/13 - 1s - 100ms/step - accuracy: 0.7474 - loss: 0.8626 - val_accuracy: 0.8095 - val_loss: 0.7233
Epoch 3/5
13/13 - 1s - 100ms/step - accuracy: 0.7964 - loss: 0.7334 - val_accuracy: 0.8571 - val_loss: 0.6299
Epoch 4/5
13/13 - 1s - 101ms/step - accuracy: 0.8325 - loss: 0.6706 - val_accuracy: 0.8929 - val_loss: 0.5656
Epoch 5/5
13/13 - 1s - 102ms/step - accuracy: 0.8351 - loss: 0.6087 - val_accuracy: 0.8810 - val_loss: 0.5187
Epoch 1/25
    val_macro_f1 = 0.8503 (melhor: 0.8503)
13/13 - 32s - 2s/step - accuracy: 0.8428 - loss: 0.5662 - val_accuracy: 0.8810 - val_loss: 0.4581
Epoch 2/25
    val_macro_f1 = 0.8909 (melhor: 0.8909)
13/13 - 14s - 1s/step - accuracy: 0.8840 - loss: 0.4842 - val_accuracy: 0.9048 - val_loss: 0.4077
Epoch 3/25
    val_macro_f1 = 0.9153 (melhor: 0.9153)
13/13

In [8]:
# ConvNeXt-Tiny
for seed in SEEDS:
    print(f"\n===== ConvNeXt-Tiny | seed {seed} =====")
    treinar("convnext_tiny", seed)
    keras.backend.clear_session()



===== ConvNeXt-Tiny | seed 0 =====
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/5
13/13 - 29s - 2s/step - accuracy: 0.3943 - loss: 1.6956 - val_accuracy: 0.5595 - val_loss: 0.9655
Epoch 2/5
13/13 - 6s - 470ms/step - accuracy: 0.6134 - loss: 1.1759 - val_accuracy: 0.7262 - val_loss: 0.7300
Epoch 3/5
13/13 - 6s - 479ms/step - accuracy: 0.6778 - loss: 0.9781 - val_accuracy: 0.7500 - val_loss: 0.6448
Epoch 4/5
13/13 - 6s - 458ms/step - accuracy: 0.7191 - loss: 0.7747 - val_accuracy: 0.7857 - val_loss: 0.5496
Epoch 5/5
13/13 - 6s - 428ms/step - accuracy: 0.7629 - loss: 0.7171 - val_accuracy: 0.8214 - val_loss: 0.4786
Epoch 1/25
    val_macro_f1 = 0.8148 (melhor: 0.8148)
13/13 - 77s - 6s/step - accuracy: 0.8041 - loss: 0.5694 - val_accuracy: 0.8452 - val_loss: 0.4053
Epoch 2/25
    val_macro_f1 = 0.8503 (melhor: 0.8503)
13/13 - 21s - 2s/step - accuracy: 0.8144 - loss: 0.6060 - val_accuracy: 0.8810 - val_loss: 0.3704
Epoch 3/25
    val_macro_f1 = 0.8602 (melhor: 0.8602)
13/13

In [9]:
# DenseNet121
for seed in SEEDS:
    print(f"\n===== DenseNet121 | seed {seed} =====")
    treinar("densenet121", seed)
    keras.backend.clear_session()



===== DenseNet121 | seed 0 =====
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/5
13/13 - 20s - 2s/step - accuracy: 0.3144 - loss: 1.4557 - val_accuracy: 0.6905 - val_loss: 1.0505
Epoch 2/5
13/13 - 2s - 178ms/step - accuracy: 0.5567 - loss: 1.1729 - val_accuracy: 0.7738 - val_loss: 0.8234
Epoch 3/5
13/13 - 2s - 178ms/step - accuracy: 0.6546 - loss: 0.9390 - val_accuracy: 0.8214 - val_loss: 0.6609
Epoch 4/5
13/13 - 3s - 193ms/step - accuracy: 0.6985 - loss: 0.9266 - val_accuracy: 0.8214 - val_loss: 0.6315
Epoch 5/5
13/13 - 2s - 190ms/step - accuracy: 0.7655 - loss: 0.7477 - val_accuracy: 0.8690 - val_loss: 0.5113
Epoch 1/25
    val_macro_f1 = 0.8571 (melhor: 0.8571)
13/13 - 49s - 4s/step - accuracy: 0.7732 - loss: 0.7088 - val_accuracy: 0.9048 - val_loss: 0.3183
Epoch 2/25
    val_macro_f1 = 0.8426 (melhor: 0.8571)
13/13 - 6s - 443ms/step - accuracy: 0.8273 - loss: 0.5365 - val_accuracy: 0.8929 - val_loss: 0.2816
Epoch 3/25
    val_macro_f1 = 0.8614 (melhor: 0.8614)
13/13 -

In [11]:
import glob
for f in sorted(glob.glob("/content/drive/MyDrive/P2_runs/preds_*_val.csv")):
    df = pd.read_csv(f)
    print(f"{f.split('/')[-1]:40s} accuracy val = {(df.y_true == df.y_pred).mean():.3f}")

preds_convnext_tiny_seed0_val.csv        accuracy val = 0.940
preds_convnext_tiny_seed1_val.csv        accuracy val = 0.893
preds_convnext_tiny_seed2_val.csv        accuracy val = 0.940
preds_densenet121_seed0_val.csv          accuracy val = 0.952
preds_densenet121_seed1_val.csv          accuracy val = 0.964
preds_densenet121_seed2_val.csv          accuracy val = 0.988
preds_efficientnetb0_seed0_val.csv       accuracy val = 0.940
preds_efficientnetb0_seed1_val.csv       accuracy val = 0.917
preds_efficientnetb0_seed2_val.csv       accuracy val = 0.952
preds_resnet50_seed0_val.csv             accuracy val = 0.988
preds_resnet50_seed1_val.csv             accuracy val = 0.964
preds_resnet50_seed2_val.csv             accuracy val = 0.976


In [12]:
# Métricas por seed e média das 3 seeds (conjunto de teste)
from sklearn.metrics import accuracy_score
linhas = []
for f in sorted(glob.glob(str(RUNS_DIR / "preds_*_test.csv"))):
    df = pd.read_csv(f)
    linhas.append({"modelo": df.modelo[0], "seed": df.seed[0],
                   "accuracy": accuracy_score(df.y_true, df.y_pred),
                   "macro_f1": f1_score(df.y_true, df.y_pred, average="macro"),
                   "macro_recall": recall_score(df.y_true, df.y_pred, average="macro")})
por_seed = pd.DataFrame(linhas)
print("Métricas por seed:")
print(por_seed.round(4).to_string(index=False))
print("\nMédia das 3 seeds por modelo:")
print(por_seed.groupby("modelo")[["accuracy", "macro_f1", "macro_recall"]].mean().round(4))


Métricas por seed:
        modelo  seed  accuracy  macro_f1  macro_recall
 convnext_tiny     0    0.9167    0.8730        0.9292
 convnext_tiny     1    0.8929    0.8483        0.9083
 convnext_tiny     2    0.8452    0.7938        0.8375
   densenet121     0    0.9286    0.8698        0.8771
   densenet121     1    0.9524    0.9182        0.9292
   densenet121     2    0.9643    0.9435        0.9688
efficientnetb0     0    0.9405    0.9152        0.9521
efficientnetb0     1    0.9405    0.9064        0.9500
efficientnetb0     2    0.9167    0.8732        0.9271
      resnet50     0    1.0000    1.0000        1.0000
      resnet50     1    0.9405    0.9059        0.9479
      resnet50     2    0.9762    0.9658        0.9792

Média das 3 seeds por modelo:
                accuracy  macro_f1  macro_recall
modelo                                          
convnext_tiny     0.8849    0.8384        0.8917
densenet121       0.9484    0.9105        0.9250
efficientnetb0    0.9325    0.8983     